# 🏆 Churn Prediction: Telco Round 2 (V9.0 - ADVANCED OPTIMIZATION)
**Dataset**: [blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
**Focus**: Tenure Binning + High-Intensity Search (n_iter=15) + Automated Threshold Retrieval (Recall >= 80%).

In [ ]:
# 1. Setup & Auth
!pip uninstall umap-learn hdbscan -y -q
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub scikit-learn==1.5.1 imbalanced-learn -q

import sklearn; print(f"✅ Environment Ready: {sklearn.__version__}")
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
import opendatasets as od
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, recall_score, precision_score, make_scorer, precision_recall_curve, f1_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Advanced Cleaning (Tenure Binning + Fillna)
def clean_telco_data(df):
    df.columns = [col.lower() for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid'] if c in df.columns])
    
    # Handle numeric conversion & Fill na instead of drop
    df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
    df['totalcharges'] = df['totalcharges'].fillna(0)
    
    # --- [Round 2 Upgrade] Tenure Binning ---
    bins = [0, 12, 24, 48, 60, 100]
    labels = ['0-12m', '12-24m', '24-48m', '48-60m', '60m+']
    df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True).astype(str)
    
    if 'churn' in df.columns:
        df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    return df

if not os.path.exists("telco-customer-churn"):
    od.download("https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

df_raw = clean_telco_data(pd.read_csv("telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"))
print(f"📊 Round 2 Dataset Prepared: {df_raw.shape}")
df_train, df_test = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])


In [ ]:
# 3. High Intensity Training Engine (V9.0)
def run_telco_round2(df_tr, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    num_f = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_f = X_tr.select_dtypes(include=['object']).columns.tolist()
    
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_f),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_f)
    ])
    
    if model_type == "xgboost":
        clf = XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42)
        m_p = {
            'clf__max_depth': [3, 4, 5, 6],
            'clf__learning_rate': [0.01, 0.05, 0.1],
            'clf__n_estimators': [100, 200, 300]
        }
    elif model_type == "random_forest":
        clf = RandomForestClassifier(random_state=42)
        m_p = {
            'clf__n_estimators': [100, 200, 300],
            'clf__max_depth': [5, 10, 15, None]
        }
    else:
        clf = LogisticRegression(max_iter=2000, random_state=42)
        m_p = {'clf__C': [0.01, 0.1, 1.0, 10.0]}
        
    mlflow.set_experiment("Churn_Round2_Colab")
    with mlflow.start_run(run_name=f"R2_COLAB_{model_type.upper()}"):
        pipe = ImbPipeline([
            ('pre', pre), 
            ('smote', SMOTE(random_state=42)), 
            ('clf', clf)
        ])
        
        # --- [Upgrade 3] High Intensity Search (n_iter=15, cv=5) ---
        search = RandomizedSearchCV(pipe, m_p, n_iter=15, cv=5, scoring='recall', verbose=1)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        
        # --- [Upgrade 2] Automated Threshold Search (Recall >= 80% Optimization) ---
        y_prb = model.predict_proba(X_te)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y_te, y_prb)
        
        best_threshold = 0.5
        valid_idx = [i for i, r in enumerate(recalls[:-1]) if r >= 0.80]
        if valid_idx:
            # Find the index with highest precision among those meeting 80% Recall
            best_sub_idx = valid_idx[0]
            max_p = 0
            for i in valid_idx:
               if precisions[i] > max_p:
                   max_p = precisions[i]
                   best_sub_idx = i
            best_threshold = thresholds[best_sub_idx]
        
        y_prd = (y_prb >= best_threshold).astype(int)
        
        print(f"\n🏆 R2 REPORT ({model_type.upper()}) | Threshold: {best_threshold:.4f}")
        print(classification_report(y_te, y_prd))
        
        mlflow.log_metric("recall", recall_score(y_te, y_prd))
        mlflow.log_metric("precision", precision_score(y_te, y_prd))
        mlflow.log_metric("f1_score", f1_score(y_te, y_prd))
        mlflow.log_param("best_threshold", best_threshold)
        
        # Fairness Auditing
        audit_df = X_te.copy()
        audit_df['predicted'] = y_prd
        gender_col = next((c for c in audit_df.columns if c.lower() == 'gender'), None)
        if gender_col:
            g_stats = audit_df.groupby(gender_col)['predicted'].mean()
            if len(g_stats) >= 2:
                mlflow.log_metric("bias_gap_gender", abs(g_stats.iloc[0] - g_stats.iloc[1]))
        
        # SHAP & Model Registration
        try:
            X_t = model.named_steps['pre'].transform(X_te)
            if hasattr(X_t, "toarray"): X_t = X_t.toarray()
            explainer = shap.Explainer(model.named_steps['clf'])
            shap_v = explainer(X_t)
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_v, X_t, feature_names=model.named_steps['pre'].get_feature_names_out(), show=False)
            plt.savefig(f"shap_r2_{model_type}.png")
            mlflow.log_artifact(f"shap_r2_{model_type}.png")
        except Exception: pass

        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_telco_round2(df_train, df_test, m)